In [14]:
import pandas as pd
import networkx as nx
import random
import json
import numpy as np

# Load data
airports = pd.read_csv("airports.dat", header=None)
routes = pd.read_csv("routes.dat", header=None)

# Set column names
airports.columns = ['Airport ID', 'Name', 'City', 'Country', 'IATA', 'ICAO',
                    'Latitude', 'Longitude', 'Altitude', 'Timezone',
                    'DST', 'Tz database time zone', 'Type', 'Source']
routes.columns = ['Airline', 'Airline ID', 'Source Airport', 'Source Airport ID',
                  'Destination Airport', 'Destination Airport ID', 'Codeshare',
                  'Stops', 'Equipment']

# Clean up invalid IDs
routes = routes.replace('\\N', pd.NA)
routes = routes.dropna(subset=['Source Airport ID', 'Destination Airport ID'])
routes['Source Airport ID'] = routes['Source Airport ID'].astype(int)
routes['Destination Airport ID'] = routes['Destination Airport ID'].astype(int)

# Create graph from routes
G = nx.Graph()
for _, row in routes.iterrows():
    G.add_edge(row['Source Airport ID'], row['Destination Airport ID'])

# Step 1: Select diverse seed airports
seed_airports = set()
countries_seen = set()
for _, row in airports.iterrows():
    if row['Country'] not in countries_seen and row['Airport ID'] in G:
        seed_airports.add(row['Airport ID'])
        countries_seen.add(row['Country'])
    if len(seed_airports) >= 5:
        break

# Step 2: Expand seeds by top 5 neighbors
sampled_airports = set(seed_airports)
for airport_id in seed_airports:
    neighbors = sorted(G[airport_id], key=lambda n: G.degree[n], reverse=True)
    sampled_airports.update(neighbors[:5])

# Step 3: Fill up to ~50 with high-degree airports
extra_airports = sorted(G.nodes, key=lambda n: G.degree[n], reverse=True)
for airport in extra_airports:
    if airport not in sampled_airports:
        sampled_airports.add(airport)
    if len(sampled_airports) >= 50:
        break

# Filter routes to sampled airports
filtered_routes = routes[
    routes['Source Airport ID'].isin(sampled_airports) &
    routes['Destination Airport ID'].isin(sampled_airports)
]

# Limit edges per airport
max_edges_per_airport = 10
limited_routes_list = []
airport_edge_count = {airport: 0 for airport in sampled_airports}

for _, row in filtered_routes.iterrows():
    source = row['Source Airport ID']
    dest = row['Destination Airport ID']
    if airport_edge_count[source] < max_edges_per_airport and airport_edge_count[dest] < max_edges_per_airport:
        limited_routes_list.append(row)
        airport_edge_count[source] += 1
        airport_edge_count[dest] += 1

limited_routes = pd.DataFrame(limited_routes_list)

# Ensure the graph is fully connected
G_limited = nx.Graph()
G_limited.add_edges_from([
    (str(row['Source Airport ID']), str(row['Destination Airport ID']))
    for _, row in limited_routes.iterrows()
])

# Keep only the largest connected component
components = list(nx.connected_components(G_limited))
largest_component = max(components, key=len)
graph_nodes = set(largest_component)

# Filter airports and routes again
filtered_airports = airports[airports['Airport ID'].astype(str).isin(graph_nodes)]
limited_routes = limited_routes[
    limited_routes['Source Airport ID'].astype(str).isin(graph_nodes) &
    limited_routes['Destination Airport ID'].astype(str).isin(graph_nodes)
]

# Prepare Graphology-style JSON
graph = {
    "attributes": {},
    "nodes": [],
    "edges": []
}

# Add nodes
for _, row in filtered_airports.iterrows():
    graph["nodes"].append({
        "key": str(row['Airport ID']),
        "attributes": {
            "name": row['Name'],
            "city": row['City'],
            "country": row['Country'],
            "IATA": row['IATA'],
            "ICAO": row['ICAO'],
            "latitude": row['Latitude'],
            "longitude": row['Longitude'],
            "timezone": row['Tz database time zone']
        }
    })

# Add edges
for _, row in limited_routes.iterrows():
    graph["edges"].append({
        "source": str(row['Source Airport ID']),
        "target": str(row['Destination Airport ID']),
        "attributes": {
            "airline": row['Airline'],
            "airline_id": row['Airline ID'],
            "codeshare": row['Codeshare'],
            "stops": row['Stops'],
            "equipment": row['Equipment']
        }
    })

# Clean attributes: replace NaNs/NA with None
def clean_attrs(d):
    return {k: (None if pd.isna(v) or v is pd.NA or isinstance(v, (pd._libs.missing.NAType, np.float64)) and pd.isna(v) else v) for k, v in d.items()}

for node in graph["nodes"]:
    node["attributes"] = clean_attrs(node["attributes"])
for edge in graph["edges"]:
    edge["attributes"] = clean_attrs(edge["attributes"])

# Export to file
with open("../data/sampled_data.json", "w") as f:
    json.dump(graph, f, indent=2)

print("✅ Exported sampled_data.json with", len(graph['nodes']), "nodes and", len(graph['edges']), "edges.")


✅ Exported sampled_data.json with 32 nodes and 152 edges.
